<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M06/M06_Lab2_LangGraph_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🕸️ M06 Lab 2 — LangGraph Agents</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐⭐ Difficulty: Advanced &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Understand <b>LangGraph</b> — graph-based agent orchestration</li>
    <li>Define <b>state</b> with TypedDict to track agent data</li>
    <li>Create <b>nodes</b> (processing functions) and <b>edges</b> (transitions)</li>
    <li>Use <b>conditional edges</b> to route between different paths</li>
    <li>Build a <b>research agent</b> that classifies, searches, calculates, and summarizes</li>
  </ol>
</div>

## 📦 Setup

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai langchain langchain-openai langgraph
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

client = setup_openai()

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Why LangGraph?</h2>
</div>

In Lab 1, we built a ReAct agent from scratch. It worked, but it was a single loop — hard to manage for complex workflows. **LangGraph** models agents as **directed graphs**:

- **Nodes** = processing steps (functions)
- **Edges** = transitions between steps
- **State** = shared data flowing through the graph
- **Conditional edges** = "if X, go to node A; otherwise go to node B"

Think of it like a **flowchart** that the agent follows, with the LLM making decisions at branch points.

| ReAct (Lab 1) | LangGraph |
|---------------|----------|
| Single loop | Graph with multiple paths |
| Hard to add complexity | Easy to add nodes/edges |
| All logic in one function | Modular, testable nodes |
| No visualization | Can visualize the graph |

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Define State</h2>
</div>

Every LangGraph agent needs a **State** — a typed dictionary that all nodes can read and write. This is how data flows through the graph.

In [ ]:
# ============================================================
# 📋 Define the Agent State
# ============================================================
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

class ResearchState(TypedDict):
    """State that flows through our research agent graph."""
    query: str                    # The user's original question
    category: str                 # Classified as: 'search', 'calculate', or 'general'
    search_result: str            # Result from the search node
    calculation_result: str       # Result from the calculator node
    final_answer: str             # The summarized final answer

# Create the LLM
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

print("✅ State and LLM defined!")
print("📋 State fields:", list(ResearchState.__annotations__.keys()))

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> <code>TypedDict</code> gives you type checking at development time. Each node receives the full state and returns a partial update — only the fields it changed.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Create Nodes</h2>
</div>

Each node is a Python function that takes the current state and returns an updated state (partial dict). Our agent has 4 nodes:

1. **Classifier** — decides if the query needs search, calculation, or general knowledge
2. **Searcher** — looks up factual information
3. **Calculator** — performs numeric computations
4. **Summarizer** — combines results into a final answer

In [ ]:
# ============================================================
# 🧩 Node 1: Classify the Query
# ============================================================

def classify_query(state: ResearchState) -> dict:
    """Classify the user's query into a category."""
    response = llm.invoke(
        f"Classify this query into exactly one category: 'search' (needs factual lookup), "
        f"'calculate' (needs math computation), or 'general' (can answer from general knowledge).\n\n"
        f"Query: {state['query']}\n\n"
        f"Reply with ONLY the category word, nothing else."
    )
    category = response.content.strip().lower().replace("'", "").replace('"', '')
    print(f"  🏷️ Classifier → '{category}'")
    return {"category": category}

print("✅ classify_query node defined")

In [ ]:
# ============================================================
# 🧩 Node 2: Search for Information
# ============================================================

def search_node(state: ResearchState) -> dict:
    """Search for factual information using the LLM."""
    response = llm.invoke(
        f"You are a research assistant. Provide a detailed, factual answer to this query. "
        f"Include specific numbers, dates, or statistics where relevant.\n\n"
        f"Query: {state['query']}"
    )
    result = response.content
    print(f"  🔍 Searcher → found {len(result)} chars of information")
    return {"search_result": result}

print("✅ search_node defined")

In [ ]:
# ============================================================
# 🧩 Node 3: Calculate
# ============================================================

def calculate_node(state: ResearchState) -> dict:
    """Perform calculations using the LLM."""
    response = llm.invoke(
        f"You are a precise calculator. Solve this step by step, showing your work.\n\n"
        f"Problem: {state['query']}\n\n"
        f"Show the calculation steps and the final answer."
    )
    result = response.content
    print(f"  🧮 Calculator → computed result")
    return {"calculation_result": result}

print("✅ calculate_node defined")

In [ ]:
# ============================================================
# 🧩 Node 4: Summarize
# ============================================================

def summarize_node(state: ResearchState) -> dict:
    """Summarize all gathered information into a final answer."""
    # Combine whatever results we have
    context = ""
    if state.get("search_result"):
        context += f"Search results: {state['search_result']}\n"
    if state.get("calculation_result"):
        context += f"Calculation: {state['calculation_result']}\n"

    response = llm.invoke(
        f"Provide a clear, concise final answer to the user's question.\n\n"
        f"Question: {state['query']}\n\n"
        f"Available information:\n{context}\n"
        f"Give a direct answer in 2-3 sentences."
    )
    print(f"  📝 Summarizer → final answer ready")
    return {"final_answer": response.content}

print("✅ summarize_node defined")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Build the Graph</h2>
</div>

Now we connect the nodes with edges. The key feature is **conditional edges** — the classifier decides which path to take.

```
START → classify → (search OR calculate OR summarize) → summarize → END
```

In [ ]:
# ============================================================
# 🕸️ Build the Graph
# ============================================================

def route_query(state: ResearchState) -> str:
    """Route to the correct node based on classification."""
    category = state.get("category", "general")
    if "search" in category:
        return "search"
    elif "calc" in category:
        return "calculate"
    else:
        return "summarize"

# Create the graph
graph = StateGraph(ResearchState)

# Add nodes
graph.add_node("classify", classify_query)
graph.add_node("search", search_node)
graph.add_node("calculate", calculate_node)
graph.add_node("summarize", summarize_node)

# Add edges
graph.add_edge(START, "classify")         # Start → Classify
graph.add_conditional_edges(              # Classify → route to correct node
    "classify",
    route_query,
    {"search": "search", "calculate": "calculate", "summarize": "summarize"}
)
graph.add_edge("search", "summarize")     # Search → Summarize
graph.add_edge("calculate", "summarize")  # Calculate → Summarize
graph.add_edge("summarize", END)          # Summarize → End

# Compile the graph into a runnable
agent = graph.compile()

print("✅ Graph compiled!")
print("📊 Nodes:", ["classify", "search", "calculate", "summarize"])
print("🔀 Conditional edge: classify → search | calculate | summarize")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Visualize the Graph</h2>
</div>

LangGraph can render the graph structure so you can see the flow visually.

In [ ]:
# ============================================================
# 📊 Visualize the Graph Structure
# ============================================================
try:
    from IPython.display import Image, display
    img_bytes = agent.get_graph().draw_mermaid_png()
    display(Image(img_bytes))
except Exception:
    # Fallback: print ASCII representation
    print("Graph structure (text):")
    print(agent.get_graph().draw_ascii())

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">6️⃣ Run the Agent</h2>
</div>

Let's test our research agent with different types of queries — each should take a different path through the graph.

In [ ]:
# ============================================================
# 🧪 Test 1: Search Query
# ============================================================
print("="*70)
print("🔍 Test 1: Search query")
print("="*70)

result = agent.invoke({"query": "What are the top 3 applications of GenAI in supply chain management?"})
print(f"\n🤖 Final Answer:\n{result['final_answer']}")

In [ ]:
# ============================================================
# 🧪 Test 2: Calculation Query
# ============================================================
print("="*70)
print("🧮 Test 2: Calculation query")
print("="*70)

result = agent.invoke({"query": "If a company processes 1,200 orders per day at $45.50 each, what is the monthly revenue (30 days)?"})
print(f"\n🤖 Final Answer:\n{result['final_answer']}")

In [ ]:
# ============================================================
# 🧪 Test 3: General Knowledge Query
# ============================================================
print("="*70)
print("💡 Test 3: General knowledge query")
print("="*70)

result = agent.invoke({"query": "Explain the difference between supervised and unsupervised learning."})
print(f"\n🤖 Final Answer:\n{result['final_answer']}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Each query took a <b>different path</b> through the graph. The classifier node acts as a "router" — deciding which specialized node should handle the query. This is the foundation of all agentic architectures: <b>classify → route → process → summarize</b>.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "In LangGraph, what is the role of 'State'?",
        "options": [
            "It stores the LLM's model weights",
            "It is a shared data structure (TypedDict) that flows through all nodes in the graph",
            "It tracks which API keys are active",
            "It records the user's conversation history only"
        ],
        "answer": 1,
        "explanation": "State is a TypedDict that all nodes can read and write. Each node receives the current state, processes it, and returns updated fields. This is how data flows through the graph."
    },
    {
        "q": "What is a 'conditional edge' in LangGraph?",
        "options": [
            "An edge that only runs if the API call succeeds",
            "An edge that runs both connected nodes in parallel",
            "An edge that routes to different nodes based on a function's return value",
            "An edge that retries failed nodes automatically"
        ],
        "answer": 2,
        "explanation": "A conditional edge uses a routing function to decide which node to go to next. In our agent, the route_query function checks the category and sends the flow to 'search', 'calculate', or directly to 'summarize'."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add a Translation Node

Add a new node called `translate_node` that translates the final answer into another language. Update the graph so that after summarizing, the agent translates the result.

Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: Add a Translation Node
# ============================================================
# Replace each ----- with the correct value

# Extended state with translation fields
class TranslateState(TypedDict):
    query: str
    category: str
    search_result: str
    calculation_result: str
    final_answer: str
    target_language: str          # New: language to translate to
    translated_answer: str        # New: the translated result

# The translation node
def translate_node(state: TranslateState) -> dict:
    """Translate the final answer to the target language."""
    lang = state.get("target_language", "Spanish")
    response = llm.invoke(
        f"Translate the following text to {lang}. Keep it natural and accurate.\n\n"
        f"{state['final_answer']}"
    )
    print(f"  🌍 Translator → translated to {lang}")
    return {"translated_answer": response.content}

# Build the extended graph
graph2 = StateGraph(------)              # What state class?

graph2.add_node("classify", classify_query)
graph2.add_node("search", search_node)
graph2.add_node("calculate", calculate_node)
graph2.add_node("summarize", summarize_node)
graph2.add_node("-----", translate_node)  # What name for the new node?

graph2.add_edge(START, "classify")
graph2.add_conditional_edges(
    "classify", route_query,
    {"search": "search", "calculate": "calculate", "summarize": "summarize"}
)
graph2.add_edge("search", "summarize")
graph2.add_edge("calculate", "summarize")
graph2.add_edge("summarize", "-----")     # Summarize → ? (new node)
graph2.add_edge("translate", -----)       # Translate → ? (end constant)

agent2 = graph2.compile()

# Test it!
result2 = agent2.invoke({
    "query": "What are three benefits of using LangGraph for AI agents?",
    "target_language": "French"
})
print(f"\n🤖 English: {result2['final_answer'][:200]}...")
print(f"\n🇫🇷 French: {result2['translated_answer'][:200]}...")

**📝 Your Observations:** *(double-click to edit this cell)*

1. How does adding a new node change the graph flow? _[Your answer]_

2. What would you change to make translation optional (only translate if a language is specified)? _[Your answer]_

3. How could you add parallel nodes (e.g., search AND calculate at the same time)? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Multi-path graph:</b> Add a "fact-check" node that verifies the search results before summarizing</li>
    <li><b>Loop:</b> Make the agent retry if the summary is too long (add a conditional edge back to summarize)</li>
    <li><b>Human-in-the-loop:</b> Add an "approval" node that pauses and asks for user confirmation before the final answer</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built a LangGraph agent with state, nodes, edges, and conditional routing.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><code style="color: #90caf9;">StateGraph</code> — define agents as directed graphs with typed state</li>
      <li><code style="color: #90caf9;">add_node()</code> — each node is a function that processes state</li>
      <li><code style="color: #90caf9;">add_conditional_edges()</code> — route dynamically based on state values</li>
      <li>LangGraph makes complex agents <b>modular, testable, and visual</b></li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M07: Hackathon (build your own agent!)</p>
</div>